# 🏦 Financial Fraud Detection Pipeline
## Notebook 1: Exploratory Data Analysis (EDA)

**Goal:** Understand the dataset structure, class imbalance, feature distributions, and correlations before modeling.

**Dataset:** [Credit Card Fraud Detection – Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

**Features:**
- `Time` – Seconds elapsed between each transaction and the first transaction
- `V1–V28` – PCA-transformed anonymized features
- `Amount` – Transaction amount
- `Class` – Target variable (1 = Fraud, 0 = Legitimate)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✅')

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. Basic Info & Missing Values

In [ ]:
print('--- Data Types & Non-Null Counts ---')
print(df.info())

print('\n--- Missing Values ---')
print(df.isnull().sum().sum(), 'total missing values')

print('\n--- Summary Statistics ---')
df.describe().T.style.background_gradient(cmap='Blues')

## 4. Class Distribution (Imbalance Check)

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('--- Class Distribution ---')
print(f'Legitimate (0): {class_counts[0]:,} ({class_pct[0]:.4f}%)')
print(f'Fraud (1):      {class_counts[1]:,} ({class_pct[1]:.4f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Transaction Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.3f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('⚠️ Highly Imbalanced Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/class_distribution.png ✅')

## 5. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cls, label, color in zip(axes, [0, 1], ['Legitimate', 'Fraud'], ['#4C72B0', '#DD8452']):
    data = df[df['Class'] == cls]['Amount']
    ax.hist(data, bins=60, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'{label} Transaction Amounts', fontsize=12, fontweight='bold')
    ax.set_xlabel('Amount ($)')
    ax.set_ylabel('Frequency')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.text(0.97, 0.95, f'Mean: ${data.mean():.2f}\nMax: ${data.max():.2f}',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8), fontsize=9)

plt.suptitle('Transaction Amount Distribution by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/amount_distribution.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/amount_distribution.png ✅')

## 6. Transaction Volume Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, cls, label, color in zip(axes, [0, 1], ['Legitimate', 'Fraud'], ['#4C72B0', '#DD8452']):
    data = df[df['Class'] == cls]
    ax.scatter(data['Time'] / 3600, data['Amount'], alpha=0.3, s=1.5, color=color)
    ax.set_ylabel('Amount ($)')
    ax.set_title(f'{label} Transactions Over Time', fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].set_xlabel('Hours Since First Transaction')
plt.suptitle('Transaction Patterns Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/time_distribution.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/time_distribution.png ✅')

## 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(18, 12))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.3, square=False,
            cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/correlation_heatmap.png ✅')

## 8. Top Features Correlated with Fraud

In [ ]:
fraud_corr = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
top_features = fraud_corr.head(10)

colors = ['#DD8452' if v > 0 else '#4C72B0' for v in top_features.values]

plt.figure(figsize=(10, 5))
bars = plt.barh(top_features.index, top_features.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Top 10 Features Correlated with Fraud', fontsize=13, fontweight='bold')
plt.xlabel('Correlation with Class (Fraud=1)')
for bar, val in zip(bars, top_features.values):
    plt.text(val + (0.002 if val > 0 else -0.002), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val > 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/top_correlated_features.png', bbox_inches='tight')
plt.show()

print('\nTop 10 features most correlated with fraud:')
print(top_features.to_string())
print('\nSaved to outputs/top_correlated_features.png ✅')

## 9. Feature Distributions: Fraud vs Legitimate (Top 6 Features)

In [ ]:
top6 = fraud_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top6):
    for cls, label, color in [(0, 'Legitimate', '#4C72B0'), (1, 'Fraud', '#DD8452')]:
        data = df[df['Class'] == cls][feat]
        ax.hist(data, bins=50, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(f'Distribution of {feat}', fontsize=11, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Top Feature Distributions: Fraud vs Legitimate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/feature_distributions.png ✅')

## 10. EDA Summary

| Finding | Detail |
|---|---|
| **Total Transactions** | 284,807 |
| **Fraud Cases** | 492 (0.17%) — severely imbalanced |
| **Missing Values** | None |
| **Key Features** | V14, V17, V12, V10, V16 most correlated with fraud |
| **Fraud Amounts** | Generally lower than legitimate transactions |
| **Imbalance Strategy** | Will use SMOTE in preprocessing notebook |

### ➡️ Next Step: `02_preprocessing.ipynb` — handle imbalance, scale features, prepare train/test splits